## Importar API da Base dos Dados



In [ ]:
!pip install basedosdados

## Controle de Versão

In [ ]:
# --- TÓPICO 0: CONFIGURAÇÕES DE AMBIENTE E REPRODUTIBILIDADE ---
import pandas as pd
import numpy as np
import pandas_gbq
import basedosdados
import matplotlib.pyplot as plt

# 1. Registro de Versões (Garante que você saiba em qual ambiente o código funcionou)
print(f"📌 Pandas: {pd.__version__}")
print(f"📌 Numpy: {np.__version__}")
print(f"📌 Pandas-GBQ: {pandas_gbq.__version__}")
print(f"📌 Matplotlib: {matplotlib.__version__}") 


# 2. Comando para gerar o requirements.txt (Rode se quiser exportar para outro ambiente)
# !pip freeze > requirements.txt

# 3. Configurações Globais do Pandas (Para evitar comportamentos inesperados)
pd.set_option('future.no_silent_downcasting', True) # Prepara para mudanças no Pandas 3.0

def preparar_populacao_referencia(df_base):
    # Reconstroi a populacao_ref a partir das colunas-base para evitar estado inconsistente
    # quando as celulas forem executadas fora de ordem.
    df_base = df_base.copy().sort_values(['id_municipio', 'ano'])

    if 'populacao_ref_bruta' in df_base.columns:
        pop_base = df_base['populacao_ref_bruta']
    else:
        pop_base = df_base['populacao_urbana']

    df_base['populacao_urbana_limpa'] = pop_base
    df_base['populacao_urbana_era_nula'] = df_base['populacao_urbana_limpa'].isna()

    df_base['populacao_ref'] = df_base['populacao_urbana_limpa']
    mask_fallback = df_base['populacao_ref'].isna()
    df_base.loc[mask_fallback, 'populacao_ref'] = df_base.loc[mask_fallback, 'populacao_atendida_agua']

    df_base['populacao_usou_fallback_agua'] = (
        df_base['populacao_urbana_era_nula'] & df_base['populacao_atendida_agua'].notna()
    )

    df_base['fonte_populacao'] = np.where(
        df_base['populacao_urbana_limpa'].notna(),
        'urbana',
        np.where(
            df_base['populacao_atendida_agua'].notna(),
            'agua',
            'missing'
        )
    )

    # Nula = ainda nula apos todos os fallbacks (para interpolacao)
    df_base['populacao_ref_era_nula'] = df_base['populacao_ref'].isna()
    df_base['populacao_ref'] = df_base.groupby('id_municipio')['populacao_ref'].transform(
        lambda x: x.interpolate(method='linear', limit=2, limit_area='inside')
    )

    return df_base


def calcular_flags_evidencia(df):
    # Recalcula tem_rede e tem_trat_real a partir das colunas-base do df recebido.
    # Evita estado global inconsistente quando células são reexecutadas fora de ordem.
    evidencias_coleta = [c for c in [
        'extensao_rede_esgoto',
        'populacao_urbana_atendida_esgoto',
        'quantidade_ligacao_ativa_esgoto'
    ] if c in df.columns]
    evidencias_tratamento = [c for c in ['volume_esgoto_tratado'] if c in df.columns]

    tem_rede = (
        df[evidencias_coleta].fillna(0).gt(0).any(axis=1)
        if evidencias_coleta else pd.Series(False, index=df.index)
    )
    tem_trat_real = (
        df[evidencias_tratamento].fillna(0).gt(0).any(axis=1)
        if evidencias_tratamento else pd.Series(False, index=df.index)
    )
    return tem_rede, tem_trat_real


## Autenticar usuário

In [ ]:
import pandas_gbq

from google.colab import auth
auth.authenticate_user()
print('Autenticado com sucesso!')

# Saneamento: Água e Esgoto

## Importar Base de Dados

In [ ]:
# Seu ID corrigido (garanta que não haja espaços antes ou depois)
import pandas_gbq

PROJECT_ID = "analise-saneamento"

# SQL otimizada: seleciona apenas colunas usadas e filtra UF/ano na origem
sql = """
SELECT
  ano,
  id_municipio,
  sigla_uf,
  quantidade_economia_residencial_ativa_agua,
  quantidade_economia_residencial_ativa_esgoto,
  quantidade_ligacao_total_agua,
  quantidade_ligacao_total_esgoto,
  populacao_urbana,
  populacao_atendida_agua,
  indice_atendimento_total_agua,
  indice_atendimento_esgoto_agua,
  indice_atendimento_urbano_agua,
  indice_tratamento_esgoto,
  indice_perda_distribuicao_agua,
  indice_consumo_agua_per_capita,
  volume_esgoto_coletado,
  volume_esgoto_tratado,
  extensao_rede_agua,
  extensao_rede_esgoto,
  populacao_atentida_esgoto AS populacao_urbana_atendida_esgoto,
  quantidade_ligacao_ativa_esgoto,
  investimento_total_municipio,
  investimento_total_estado,
  investimento_total_prestador,
  despesa_exploracao,
  arrecadacao_total,
  receita_operacional
FROM `basedosdados.br_mdr_snis.municipio_agua_esgoto`
WHERE sigla_uf = 'ES' AND ano >= 2006
"""

try:
    df = pandas_gbq.read_gbq(sql, project_id=PROJECT_ID)
    print("Sucesso! Dados do ES carregados.")
    display(df.head())
except Exception as e:
    raise RuntimeError(f"Erro ao acessar a tabela de saneamento: {e}")


## Definindo estrutura do data frame

In [ ]:
colunas_selecionadas = [
    # Identificação
    'ano', 'id_municipio', 'sigla_uf',

    # Para identificar comunidades vulneráveis
    'quantidade_economia_residencial_ativa_agua',
    'quantidade_economia_residencial_ativa_esgoto',
    'quantidade_ligacao_total_agua',
    'quantidade_ligacao_total_esgoto',

    # Demografia e Cobertura
    'populacao_urbana',
    'populacao_atendida_agua',
    'indice_atendimento_total_agua',
    'indice_atendimento_esgoto_agua',
    'indice_atendimento_urbano_agua',

    # Qualidade e Performance
    'indice_tratamento_esgoto',
    'indice_perda_distribuicao_agua',
    'indice_consumo_agua_per_capita',
    'volume_esgoto_coletado',
    'volume_esgoto_tratado',

    # Infraestrutura (EVIDÊNCIAS PARA O SCRIPT)
    'extensao_rede_agua',
    'extensao_rede_esgoto',
    'populacao_urbana_atendida_esgoto',
    'quantidade_ligacao_ativa_esgoto',

    # Investimentos
    'investimento_total_municipio',
    'investimento_total_estado',
    'investimento_total_prestador',
    'despesa_exploracao',
    'arrecadacao_total',
    'receita_operacional'
]

df_silver = df[colunas_selecionadas].copy()

# Asserção para garantir que o filtro do SQL foi aplicado
assert df_silver['ano'].min() >= 2006, "Dado fora do intervalo esperado — verificar SQL"
print(f"Novo tamanho do dataset: {df_silver.shape}")


## Limpeza dos Dados

In [ ]:
# --- TRATAMENTO DE POPULAÇÃO E ESGOTO ---
df_gold = df_silver.sort_values(['id_municipio', 'ano']).copy()

# snapshot pré-imputação para auditoria/validação posterior
df_gold['populacao_ref_bruta'] = df_gold['populacao_urbana']
mask_pop_urbana_zero = df_gold['populacao_ref_bruta'].eq(0)
df_gold.loc[mask_pop_urbana_zero, 'populacao_ref_bruta'] = np.nan

df_gold['populacao_atendida_agua'] = df_gold['populacao_atendida_agua'].replace(0, np.nan)

# 2. Lógica Rigorosa de Evidência (Coleta vs Tratamento) — via função para evitar estado global
tem_rede, tem_trat_real = calcular_flags_evidencia(df_gold)


mask_coleta_nulo = df_gold['indice_atendimento_esgoto_agua'].isna()
df_gold.loc[mask_coleta_nulo & ~tem_rede, 'indice_atendimento_esgoto_agua'] = 0.0

mask_trat_nulo = df_gold['indice_tratamento_esgoto'].isna()
df_gold.loc[mask_trat_nulo & ~tem_trat_real, 'indice_tratamento_esgoto'] = 0.0

mask_vol_nulo = df_gold['volume_esgoto_tratado'].isna()
df_gold.loc[mask_vol_nulo & ~tem_trat_real, 'volume_esgoto_tratado'] = 0.0

# 3. Interpolação conservadora
cols_interp = [
    'indice_atendimento_esgoto_agua',
    'indice_tratamento_esgoto',
    'volume_esgoto_tratado',
    'volume_esgoto_coletado'
]

for col in cols_interp:
    df_gold[col] = df_gold.groupby('id_municipio')[col].transform(
        lambda x: x.interpolate(method='linear', limit=2, limit_area='inside')
    )

# preencher zero apenas quando a ausência representa inexistência estrutural comprovada
df_gold.loc[df_gold['indice_atendimento_esgoto_agua'].isna() & ~tem_rede, 'indice_atendimento_esgoto_agua'] = 0.0
df_gold.loc[df_gold['indice_tratamento_esgoto'].isna() & ~tem_trat_real, 'indice_tratamento_esgoto'] = 0.0
df_gold.loc[df_gold['volume_esgoto_tratado'].isna() & ~tem_trat_real, 'volume_esgoto_tratado'] = 0.0
df_gold.loc[df_gold['volume_esgoto_coletado'].isna() & ~tem_rede, 'volume_esgoto_coletado'] = 0.0

# flags de completude para evitar tratar dado ausente como zero real
df_gold['flag_insumos_esgoto_incompletos'] = df_gold[cols_interp].isna().any(axis=1)

# 4. Cálculo dos Gaps (com proteção contra NaN)
df_gold['Volume_Esgoto_Nao_Tratado_m3'] = np.where(
    df_gold['volume_esgoto_coletado'].notna() & df_gold['volume_esgoto_tratado'].notna(),
    (df_gold['volume_esgoto_coletado'] - df_gold['volume_esgoto_tratado']).clip(lower=0).round(2),
    np.nan
)

indice_trat_limite = df_gold['indice_tratamento_esgoto'].clip(upper=100)

df_gold['Atendimento_Com_Tratamento_Efetivo_Percentual'] = np.where(
    df_gold['indice_atendimento_esgoto_agua'].notna() & indice_trat_limite.notna(),
    (df_gold['indice_atendimento_esgoto_agua'] * (indice_trat_limite / 100)).round(2),
    np.nan
)

df_gold['Deficit_Cobertura_Tratamento_Percentual'] = np.where(
    df_gold['indice_atendimento_esgoto_agua'].notna() & df_gold['Atendimento_Com_Tratamento_Efetivo_Percentual'].notna(),
    (df_gold['indice_atendimento_esgoto_agua'] - df_gold['Atendimento_Com_Tratamento_Efetivo_Percentual']).clip(lower=0).round(2),
    np.nan
)

# 5. Investimentos (não assumir zero quando todos os componentes estiverem ausentes)
cols_invest = ['investimento_total_municipio', 'investimento_total_estado', 'investimento_total_prestador']
df_gold['investimento_total_consolidado'] = df_gold[cols_invest].sum(axis=1, min_count=1)
df_gold['flag_investimento_parcial_ou_ausente'] = df_gold[cols_invest].isna().any(axis=1)

print("Processamento final concluído! Base limpa com interpolação conservadora.")

# 6. População de referência
df_gold = preparar_populacao_referencia(df_gold)
print("✅ População de referência preparada para o cálculo de morbidade.")

## Adicionais para calcular índícios

In [ ]:
# --- ADICIONAIS DE EXCELÊNCIA ---

# 1. Criação das Flags de Qualidade (Para transparência analítica)
tem_rede_flags, _ = calcular_flags_evidencia(df_gold)


df_gold['qualidade_dados_esgoto'] = 'dados_preenchidos'

# Caso de possível omissão: Tem rede, mas o tratamento resultou em 0
df_gold.loc[tem_rede_flags & (df_gold['indice_tratamento_esgoto'] == 0),
            'qualidade_dados_esgoto'] = 'possivel_omissao_informativa'

# Distingue ausência de dados de inexistência provável
mask_sem_rede_atendimento_zero = (
    (df_gold['extensao_rede_esgoto'] == 0) &
    (df_gold['indice_atendimento_esgoto_agua'] == 0)
)

df_gold.loc[
    mask_sem_rede_atendimento_zero & (~df_gold['flag_insumos_esgoto_incompletos']),
    'qualidade_dados_esgoto'
 ] = 'provavelmente_inexistente'

df_gold.loc[
    mask_sem_rede_atendimento_zero & df_gold['flag_insumos_esgoto_incompletos'],
    'qualidade_dados_esgoto'
 ] = 'ausencia_de_dados'

# 2. Limpeza de resíduos de arredondamento no novo Déficit (CORRIGIDO)
# Removemos apenas ruído de ponto flutuante (ex: 1e-12)
df_gold.loc[df_gold['Deficit_Cobertura_Tratamento_Percentual'] < 1e-6,
            'Deficit_Cobertura_Tratamento_Percentual'] = 0

print("💎 Finalização concluída: Flags adicionadas e Déficit limpo!")

## Limpeza dos NAs

In [ ]:
# --- LIMPEZA FINAL DE INFRAESTRUTURA (Ajuste técnico) ---

colunas_contagem = [
    'extensao_rede_esgoto',
    'populacao_urbana_atendida_esgoto',
    'quantidade_ligacao_ativa_esgoto',
    'quantidade_economia_residencial_ativa_esgoto',
    'quantidade_ligacao_total_esgoto'
]

for col in colunas_contagem:
    if col in df_gold.columns:
        # Preserva NaN para análise e cria versão para exibição/joins leves
        df_gold[f'{col}_display'] = df_gold[col].fillna(0)

# Recalcula evidência de rede sem destruir os NaNs originais — via função centralizada
tem_rede_pos_display, _ = calcular_flags_evidencia(df_gold)


print("✅ Colunas _display criadas; NaNs originais preservados para diagnóstico.")


## Integração de Dados de Saúde (TabNet)

In [ ]:
# --- CÉLULA 7: INTEGRAÇÃO DATASUS E CÁLCULO DE PESOS EPIDEMIOLÓGICOS ---

# Pesos versionados para evitar deriva histórica silenciosa no dashboard
VERSAO_PESOS = '2025-04'
W_AGUA_CALCULADO = 0.52
W_ESGOTO_CALCULADO = 0.48
RECALCULAR_PESOS = False  # mude para True apenas ao atualizar os CSVs

# Usa pesos fixos por padrão
W_AGUA = W_AGUA_CALCULADO
W_ESGOTO = W_ESGOTO_CALCULADO
print(f"⚖️ Pesos fixos (versão {VERSAO_PESOS}): W_AGUA={W_AGUA}, W_ESGOTO={W_ESGOTO}")

# defaults defensivos para evitar quebra em cascata
df_final = df_gold.copy()

def processar_saude(arquivo, nome_metrica):
    # Detecta dinamicamente faixa util e evita dependencia de skipfooter fixo
    df_raw = pd.read_csv(
        arquivo,
        sep=';',
        encoding='iso-8859-1',
        header=None,
        dtype=str
    )

    col0 = df_raw.iloc[:, 0].fillna('').astype(str).str.strip()
    mask_id = col0.str.match(r'^\d{6,}')

    if not mask_id.any():
        raise ValueError(f'Não foi possível identificar linhas de municípios em {arquivo}.')

    primeiro_dado = int(mask_id.idxmax())
    inicio = max(primeiro_dado - 1, 0)  # linha de cabeçalho logo antes do primeiro município

    apos_primeiro = col0.iloc[primeiro_dado + 1:]
    mask_fim = ~apos_primeiro.str.match(r'^\d{6,}')

    # idxmax() retorna o PRIMEIRO True encontrado, não necessariamente o último.
    # Isso é o comportamento correto: queremos parar na primeira linha que NÃO
    # seja um município válido (ex: rodapé "Total", linha vazia, nota de rodapé).
    # Se todas as linhas forem municípios válidos (mask_fim.all() is False),
    # usamos len(df_raw) como sentinela de fim do arquivo.
    indices_fim = mask_fim.to_numpy().nonzero()[0]
    fim = int(apos_primeiro.index[indices_fim[0]]) if len(indices_fim) > 0 else len(df_raw)


    nrows = max(fim - inicio - 1, 1)

    df_s = pd.read_csv(
        arquivo,
        sep=';',
        encoding='iso-8859-1',
        skiprows=inicio,
        nrows=nrows
    )

    df_s = df_s.drop(columns=['Total', 'total'], errors='ignore')
    col_mun = df_s.columns[0]
    df_s = df_s[df_s[col_mun].astype(str).str.extract(r'^(\d{6})')[0].notna()].copy()

    df_long = df_s.melt(
        id_vars=[col_mun],
        var_name='ano_bruto',
        value_name=nome_metrica
    )

    df_long['id_municipio_6'] = df_long[col_mun].astype(str).str.extract(r'^(\d{6})')[0].str.strip()
    df_long['ano'] = pd.to_numeric(df_long['ano_bruto'].astype(str).str.extract(r'(\d{4})')[0], errors='coerce')

    # conversão robusta de contagem
    df_long[nome_metrica] = (
        df_long[nome_metrica]
        .astype(str)
        .str.replace('.', '', regex=False)
        .str.replace(',', '.', regex=False)
        .str.strip()
        .replace({'-': None, '': None, 'nan': None})
    )
    df_long[nome_metrica] = pd.to_numeric(df_long[nome_metrica], errors='coerce')

    df_limpo = df_long.dropna(subset=['ano', 'id_municipio_6']).copy()
    df_limpo['ano'] = df_limpo['ano'].astype(int)
    df_limpo[nome_metrica] = df_limpo[nome_metrica].fillna(0)

    return df_limpo[['id_municipio_6', 'ano', nome_metrica]]

linhas_antes = df_gold.shape[0]
df_gold['id_municipio_6'] = df_gold['id_municipio'].astype(str).str.slice(0, 6).str.strip()

# ✅ Validação de unicidade no SNIS (lado esquerdo do merge)
dupes_snis = df_gold.duplicated(subset=['id_municipio_6', 'ano'], keep=False)
if dupes_snis.any():
    exemplos = (
        df_gold.loc[dupes_snis, ['id_municipio', 'id_municipio_6', 'ano']]
        .drop_duplicates()
        .sort_values(['id_municipio_6', 'ano'])
        .head(20)
    )
    raise ValueError(
        "Colisão/duplicidade no SNIS para a chave (id_municipio_6, ano). "
        f"Exemplos:\n{exemplos}"
    )

try:
    df_s_agua = processar_saude('saude_agua_es.csv', 'internacoes_agua')
    df_s_esgoto = processar_saude('saude_esgoto_es.csv', 'internacoes_esgoto')

    # ✅ Validação de unicidade na saúde (água)
    dupes_saude_agua = df_s_agua.duplicated(subset=['id_municipio_6', 'ano'], keep=False)
    if dupes_saude_agua.any():
        exemplos = (
            df_s_agua.loc[dupes_saude_agua, ['id_municipio_6', 'ano']]
            .drop_duplicates()
            .sort_values(['id_municipio_6', 'ano'])
            .head(20)
        )
        raise ValueError(
            "Colisão/duplicidade na base de saúde (água) para a chave (id_municipio_6, ano). "
            f"Exemplos:\n{exemplos}"
        )

    # ✅ Validação de unicidade na saúde (esgoto)
    dupes_saude_esgoto = df_s_esgoto.duplicated(subset=['id_municipio_6', 'ano'], keep=False)
    if dupes_saude_esgoto.any():
        exemplos = (
            df_s_esgoto.loc[dupes_saude_esgoto, ['id_municipio_6', 'ano']]
            .drop_duplicates()
            .sort_values(['id_municipio_6', 'ano'])
            .head(20)
        )
        raise ValueError(
            "Colisão/duplicidade na base de saúde (esgoto) para a chave (id_municipio_6, ano). "
            f"Exemplos:\n{exemplos}"
        )

    anos_agua = set(df_s_agua['ano'])
    anos_esgoto = set(df_s_esgoto['ano'])
    anos_comuns = anos_agua.intersection(anos_esgoto)

    print("\n📊 DIAGNÓSTICO DE COBERTURA")
    print(f"Água: {min(anos_agua)}–{max(anos_agua)} ({len(anos_agua)} anos)")
    print(f"Esgoto: {min(anos_esgoto)}–{max(anos_esgoto)} ({len(anos_esgoto)} anos)")

    if anos_comuns:
        print(f"Interseção: {min(anos_comuns)}–{max(anos_comuns)} ({len(anos_comuns)} anos)")
    else:
        print("Interseção: nenhuma. Mantidos pesos fixos versionados.")

    if RECALCULAR_PESOS:
        if anos_comuns:
            df_agua_common = df_s_agua[df_s_agua['ano'].isin(anos_comuns)]
            df_esgoto_common = df_s_esgoto[df_s_esgoto['ano'].isin(anos_comuns)]

            total_a = float(df_agua_common['internacoes_agua'].sum())
            total_e = float(df_esgoto_common['internacoes_esgoto'].sum())
            total_g = total_a + total_e

            if total_g > 0:
                W_AGUA = total_a / total_g
                W_ESGOTO = total_e / total_g
                print(f"🔄 Pesos recalculados com base na interseção ({min(anos_comuns)}–{max(anos_comuns)}).")
            else:
                print("⚠️ Soma de internações na interseção igual a zero. Mantidos pesos fixos.")
        else:
            print("⚠️ Sem interseção entre arquivos. Mantidos pesos fixos.")

    print(f"\n⚖️ Pesos utilizados:")
    print(f"W_AGUA: {W_AGUA:.3f}")
    print(f"W_ESGOTO: {W_ESGOTO:.3f}")

    # ✅ Merges com validação estrutural
    df_final = pd.merge(
        df_gold, df_s_agua, on=['id_municipio_6', 'ano'], how='left', validate='many_to_one'
    )
    df_final = pd.merge(
        df_final, df_s_esgoto, on=['id_municipio_6', 'ano'], how='left', validate='many_to_one'
    )

    assert df_final.shape[0] == linhas_antes, (
        f"⚠️ Erro no Merge! Linhas antes: {linhas_antes}, linhas atuais: {df_final.shape[0]}"
    )

    df_final['tem_dado_saude_agua'] = df_final['internacoes_agua'].notna()
    df_final['tem_dado_saude_esgoto'] = df_final['internacoes_esgoto'].notna()
    df_final['tem_dado_saude'] = df_final['tem_dado_saude_agua'] | df_final['tem_dado_saude_esgoto']

    df_final[['internacoes_agua', 'internacoes_esgoto']] = (
        df_final[['internacoes_agua', 'internacoes_esgoto']]
        .fillna(0)
        .round()
        .astype(int)
    )

    df_final['Taxa_Morbidade_100k_Hab'] = pd.to_numeric(
        np.where(
            (df_final['populacao_ref'] > 0) & df_final['tem_dado_saude'],
            ((df_final['internacoes_agua'] + df_final['internacoes_esgoto']) / df_final['populacao_ref']) * 100000,
            np.nan
        ),
        errors='coerce'
    )

    print("\n✅ Integração concluída com pesos robustos!")

except Exception as e:
    print(f"❌ Erro crítico na Célula 7: {e}")
    print("⚠️ Pipeline continuará com df_final = df_gold e pesos default 0.5/0.5.")
    df_final = df_gold.copy()
    if 'internacoes_agua' not in df_final.columns:
        df_final['internacoes_agua'] = 0
    if 'internacoes_esgoto' not in df_final.columns:
        df_final['internacoes_esgoto'] = 0
    df_final['tem_dado_saude_agua'] = False
    df_final['tem_dado_saude_esgoto'] = False
    df_final['tem_dado_saude'] = False
    if 'Taxa_Morbidade_100k_Hab' not in df_final.columns:
        df_final['Taxa_Morbidade_100k_Hab'] = np.nan

## Engenharia de Risco Social Multidimensional (Água vs. Esgoto)

In [ ]:
# --- CÉLULA 8: ENGENHARIA DO RISCO SOCIAL FINAL (VERSÃO FINAL ROBUSTA) ---

# 1. Conversão numérica
cols_infra = [
    'indice_atendimento_total_agua',
    'indice_atendimento_esgoto_agua',
    'indice_tratamento_esgoto'
]

for col in cols_infra:
    df_final[col] = pd.to_numeric(df_final[col], errors='coerce')

df_final['Taxa_Morbidade_100k_Hab'] = pd.to_numeric(
    df_final['Taxa_Morbidade_100k_Hab'], errors='coerce'
 )
df_final['arrecadacao_total'] = pd.to_numeric(df_final['arrecadacao_total'], errors='coerce')
df_final['receita_operacional'] = pd.to_numeric(df_final['receita_operacional'], errors='coerce')

# 2. Déficit de água (não tratar NaN como cobertura zero)
df_final['indice_atendimento_total_agua'] = df_final['indice_atendimento_total_agua'].clip(0, 100)
df_final['def_agua'] = np.where(
    df_final['indice_atendimento_total_agua'].notna(),
    (100 - df_final['indice_atendimento_total_agua']).clip(0, 100),
    np.nan
)

# 3. Déficit de esgoto (não tratar NaN como ausência total)
df_final['indice_atendimento_esgoto_agua'] = df_final['indice_atendimento_esgoto_agua'].clip(0, 100)
df_final['indice_tratamento_esgoto'] = df_final['indice_tratamento_esgoto'].clip(0, 100)

# Coluna intermediária _calc: permite auditar def_esgoto sem poluir a saída final
df_final['eficiencia_esgoto_calc'] = np.where(
    df_final['indice_atendimento_esgoto_agua'].notna() & df_final['indice_tratamento_esgoto'].notna(),
    df_final['indice_atendimento_esgoto_agua'] * (df_final['indice_tratamento_esgoto'] / 100),
    np.nan
)

df_final['def_esgoto'] = np.where(
    df_final['eficiencia_esgoto_calc'].notna(),
    (100 - df_final['eficiencia_esgoto_calc']).clip(0, 100),
    np.nan
)


# 4. Vazio sanitário com reponderação pelos componentes disponíveis
peso_agua_disp = np.where(df_final['def_agua'].notna(), float(W_AGUA), 0.0)
peso_esgoto_disp = np.where(df_final['def_esgoto'].notna(), float(W_ESGOTO), 0.0)
peso_total_disp = peso_agua_disp + peso_esgoto_disp

df_final['vazio_sanitario'] = np.where(
    peso_total_disp > 0,
    ((df_final['def_agua'].fillna(0) * peso_agua_disp) + (df_final['def_esgoto'].fillna(0) * peso_esgoto_disp)) / peso_total_disp,
    np.nan
)
df_final['flag_insumos_risco_incompletos'] = df_final[['def_agua', 'def_esgoto']].isna().any(axis=1)

# 5. Índice combinado em escala única (0-100)
df_final['sem_dados_saude'] = df_final['Taxa_Morbidade_100k_Hab'].isna()

vs = df_final['vazio_sanitario'].fillna(0)
taxa = df_final['Taxa_Morbidade_100k_Hab']
max_taxa = taxa.max()

if pd.notna(max_taxa) and max_taxa > 0:
    taxa_norm = np.where(taxa.notna(), (taxa / max_taxa) * 100, 0.0)
else:
    taxa_norm = np.zeros(len(df_final), dtype=float)

# Critério: literatura de saneamento básico (ex: PLANSAB 2019) atribui
# peso maior à infraestrutura por sua causalidade direta; dado epidemiológico
# entra como amplificador de risco já manifesto.
PESO_SAUDE_COM_DADO = 0.4  # 40% morbidade / 60% infraestrutura
PESO_SAUDE_SEM_DADO = 0.0  # sem dado epidemiológico: 100% infraestrutura

peso_saude = np.where(df_final['sem_dados_saude'], PESO_SAUDE_SEM_DADO, PESO_SAUDE_COM_DADO)
peso_sanitario = 1.0 - peso_saude

df_final['indice_combinado'] = (vs * peso_sanitario + taxa_norm * peso_saude).round(2)
df_final['RISCO_SOCIAL_FINAL'] = df_final['indice_combinado'].round(2)

# 6. Eficiência financeira com rastreabilidade de truncamento
df_final['eficiencia_arrecadacao_bruta'] = np.where(
    df_final['receita_operacional'] > 0,
    (df_final['arrecadacao_total'] / df_final['receita_operacional']) * 100,
    np.nan
)
df_final['flag_eficiencia_arrecadacao_truncada'] = df_final['eficiencia_arrecadacao_bruta'] > 150
# Em bases SNIS, valores acima de 150% costumam indicar erro de lançamento (ex: receita subreportada).
df_final['eficiencia_arrecadacao'] = df_final['eficiencia_arrecadacao_bruta'].clip(upper=150).round(2)

print("💎 Risco Social Final calculado (modelo robusto)")
print("📊 Escala: 0 a 100")
print("🧠 Proteções: ausência de dados + controle de escala + fallback estrutural")
print("🔎 Flags de truncamento em eficiencia_arrecadacao registradas para auditoria")

# Comunidades vulneráveis

## Diagnóstico de Vetores Dominantes

In [ ]:

# --- DIAGNÓSTICO VETORIZADO DO VETOR DOMINANTE (ROBUSTO A NaN) ---

if 'internacoes_agua' in df_final.columns:
    internacoes_agua = pd.Series(
        pd.to_numeric(df_final['internacoes_agua'], errors='coerce'),
        index=df_final.index
    )
else:
    internacoes_agua = pd.Series(np.nan, index=df_final.index)

if 'internacoes_esgoto' in df_final.columns:
    internacoes_esgoto = pd.Series(
        pd.to_numeric(df_final['internacoes_esgoto'], errors='coerce'),
        index=df_final.index
    )
else:
    internacoes_esgoto = pd.Series(np.nan, index=df_final.index)

if 'tem_dado_saude' in df_final.columns:
    tem_dado_saude = df_final['tem_dado_saude'].fillna(False)
else:
    tem_dado_saude = internacoes_agua.notna() | internacoes_esgoto.notna()

# Sem dado segue apenas a flag consolidada (internações já chegam inteiras da Célula 19).
sem_dados = ~tem_dado_saude
ambos_zero = (~sem_dados) & (internacoes_agua == 0) & (internacoes_esgoto == 0)
vetor_agua = (~sem_dados) & (internacoes_agua > internacoes_esgoto)
empate = (~sem_dados) & (internacoes_agua == internacoes_esgoto) & (internacoes_agua > 0)
vetor_esgoto = (~sem_dados) & (internacoes_agua < internacoes_esgoto)

condicoes = [
    sem_dados,
    ambos_zero,
    vetor_agua,
    empate,
    vetor_esgoto
]

escolhas = [
    'Sem Dados',
    'Baixo Impacto',
    'Vetor Água',
    'Empate',
    'Vetor Esgoto'
]

df_final['vetor_dominante_doenca'] = np.select(condicoes, escolhas, default='Inconsistente')

print('📊 Perfil epidemiológico vetorizado (ES 2022):')
print(df_final[df_final['ano'] == 2022]['vetor_dominante_doenca'].value_counts())
print("⚠️ Registros marcados como 'Inconsistente':", (df_final['vetor_dominante_doenca'] == 'Inconsistente').sum())

# Validações

In [ ]:
# --- VALIDAÇÃO DA POPULAÇÃO DE REFERÊNCIA ---

def validar_populacao(df):
    df = df.sort_values(['id_municipio', 'ano']).copy()

    relatorio = {}

    # 1. Valores inválidos
    invalidos = df[
        (df['populacao_ref'].isna()) |
        (df['populacao_ref'] <= 0)
    ]
    relatorio['valores_invalidos'] = invalidos

    # 2. Crescimento ano a ano (%)
    df['populacao_anterior'] = df.groupby('id_municipio')['populacao_ref'].shift(1)
    df['crescimento_pct'] = (
        (df['populacao_ref'] - df['populacao_anterior']) /
        df['populacao_anterior']
    ) * 100

    crescimento_absurdo = df[
        df['crescimento_pct'].abs() > 20
    ]
    relatorio['crescimento_absurdo'] = crescimento_absurdo

    # 3. Outliers (z-score por município)

    df['outlier'] = df.groupby('id_municipio', group_keys=False)['populacao_ref'].transform(
        lambda s: ((s - s.mean()) / s.std()).abs().gt(3) if pd.notna(s.std()) and s.std() != 0 else pd.Series(False, index=s.index)
    )
    relatorio['outliers'] = df[df['outlier']]

    # 4. Saltos absolutos grandes
    df['delta_abs'] = (df['populacao_ref'] - df['populacao_anterior']).abs()

    salto_grande = df[
        df['delta_abs'] > df['populacao_ref'] * 0.15
    ]
    relatorio['saltos_grandes'] = salto_grande

    # 5. Lacunas originalmente imputadas
    if 'populacao_ref_era_nula' in df.columns:
        relatorio['valores_imputados'] = df[df['populacao_ref_era_nula']]
    else:
        relatorio['valores_imputados'] = df.iloc[0:0].copy()

    print("\n📊 RELATÓRIO DE VALIDAÇÃO DA POPULAÇÃO\n")
    print(f"❌ Valores inválidos: {len(relatorio['valores_invalidos'])}")
    print(f"📈 Crescimentos suspeitos (>20%): {len(relatorio['crescimento_absurdo'])}")
    print(f"📊 Outliers estatísticos: {len(relatorio['outliers'])}")
    print(f"⚠️ Saltos absolutos grandes: {len(relatorio['saltos_grandes'])}")
    print(f"🩹 Valores imputados/interpolados: {len(relatorio['valores_imputados'])}")

        # Apelidos e coluna adicional para células de classificação posteriores usarem sem recalcular
    df['pop_ant'] = df['populacao_anterior']
    df['pop_prox'] = df.groupby('id_municipio')['populacao_ref'].shift(-1)

    return relatorio, df

# Valida a base que será efetivamente exportada
relatorio, df_validado = validar_populacao(df_final)

In [ ]:
# municípios mais problemáticos
relatorio['crescimento_absurdo']['id_municipio'].value_counts().head(10)

In [ ]:
relatorio['outliers']['id_municipio'].value_counts().head(10)

In [ ]:
relatorio['saltos_grandes']['id_municipio'].value_counts().head(10)

In [ ]:
def analisar_municipio(df, municipio_id):

    df_m = df[df['id_municipio'].astype(str) == str(municipio_id)].sort_values('ano')

    print(df_m[['ano', 'populacao_ref', 'crescimento_pct']])

    plt.plot(df_m['ano'], df_m['populacao_ref'], marker='o')
    plt.title(f"Município {municipio_id}")
    plt.grid()
    plt.show()


In [ ]:
analisar_municipio(df_validado, '3200359')

In [ ]:
df_validado = df_validado.sort_values(['id_municipio', 'ano'])

df_validado['erro_pontual'] = (
    (df_validado['crescimento_pct'].abs() > 30) &
    (
        (df_validado['pop_prox'] - df_validado['populacao_ref']).abs() >
        (df_validado['populacao_ref'] * 0.2)
    )
)

df_validado['mudanca_base'] = (
    (df_validado['crescimento_pct'] > 30) &
    (df_validado['pop_prox'] > df_validado['populacao_ref'] * 0.9)
)

# usa marcação real de imputação original, não NaN após preenchimento
if 'populacao_ref_era_nula' in df_validado.columns:
    df_validado['buraco_antes'] = df_validado.groupby('id_municipio')['populacao_ref_era_nula'].transform(
        lambda x: x.astype(int).rolling(2, min_periods=1).sum()
    )
else:
    df_validado['buraco_antes'] = 0

df_validado['erro_interpolacao'] = (
    (df_validado['crescimento_pct'].abs() > 20) &
    (df_validado['buraco_antes'] > 0)
)

df_validado['municipio_pequeno'] = df_validado['populacao_ref'] < 20000

df_validado['variacao_pequeno'] = (
    df_validado['municipio_pequeno'] &
    (df_validado['crescimento_pct'].abs() > 20)
)

df_validado['erro_pontual'] = df_validado['erro_pontual'].fillna(False)
df_validado['mudanca_base'] = df_validado['mudanca_base'].fillna(False)
df_validado['erro_interpolacao'] = df_validado['erro_interpolacao'].fillna(False)
df_validado['variacao_pequeno'] = df_validado['variacao_pequeno'].fillna(False)


In [ ]:
# --- Classificação vetorizada (np.select) — O(n) sem overhead de Python puro ---
condicoes_erro = [
    df_validado['erro_pontual'],
    df_validado['mudanca_base'],
    df_validado['erro_interpolacao'],
    df_validado['variacao_pequeno'],
]
escolhas_erro = ['erro_pontual', 'mudanca_base', 'interpolacao_ruim', 'municipio_pequeno']
df_validado['tipo_erro'] = np.select(condicoes_erro, escolhas_erro, default='ok')


# Propaga flags de qualidade para a base final exportável
cols_flags_pop = [
    'id_municipio',
    'ano',
    'erro_pontual',
    'mudanca_base',
    'erro_interpolacao',
    'variacao_pequeno',
    'tipo_erro'
 ]
cols_existentes = [c for c in cols_flags_pop if c in df_validado.columns]

if len(cols_existentes) >= 3:
    flags_para_merge = df_validado[cols_existentes].copy()

    cols_regrava = [c for c in cols_existentes if c not in ['id_municipio', 'ano']]
    df_final = df_final.drop(columns=cols_regrava, errors='ignore')
    df_final = df_final.merge(flags_para_merge, on=['id_municipio', 'ano'], how='left')

    # default seguro para linhas sem classificação explícita
    if 'tipo_erro' in df_final.columns:
        df_final['tipo_erro'] = df_final['tipo_erro'].fillna('ok')

    for col_bool in ['erro_pontual', 'mudanca_base', 'erro_interpolacao', 'variacao_pequeno']:
        if col_bool in df_final.columns:
            df_final[col_bool] = df_final[col_bool].fillna(False).astype(bool)

    print('✅ Flags de qualidade da população sincronizadas em df_final.')
else:
    print('⚠️ Colunas de flags não encontradas em df_validado para sincronização.')

In [ ]:
df_validado['tipo_erro'].value_counts()

In [ ]:
df_validado[df_validado['tipo_erro'] != 'ok'][
    ['id_municipio', 'ano', 'populacao_ref', 'tipo_erro']
]

In [ ]:
df_validado[df_validado['tipo_erro'] == 'mudanca_base']['id_municipio'].unique()

# Exportação

In [ ]:
# --- EXPORTAÇÃO DE ALTA PERFORMANCE (PARQUET) ---

# O formato Parquet preserva os tipos de dados (int, float, datetime)
# e oferece compressão superior ao CSV, ideal para o consumo no Streamlit.

try:
    df_final.to_parquet('base_diamante_es_vfinal.parquet', index=False)
    print(f"🚀 Projeto finalizado com sucesso!")
    print(f"📦 Arquivo 'base_diamante_es_vfinal.parquet' gerado.")
    print(f"📊 Total de registros: {df_final.shape[0]}")
    print(f"💾 Tipos preservados: Internações permanecem como INT, Índices como FLOAT.")
except Exception as e:
    print(f"❌ Erro ao exportar para Parquet: {e}")
    # Fallback de segurança para CSV caso o ambiente falte dependência (pyarrow/fastparquet)
    df_final.to_csv('base_diamante_es_vfinal.csv', sep=';', index=False, encoding='utf-8-sig')
    print("⚠️ Exportado para CSV como alternativa de segurança.")


In [ ]:
# Listar todas as colunas do dataset original (df)
print("Colunas do DataFrame 'df':")
display(df.columns.tolist())